<a href="https://colab.research.google.com/github/abelunbound/fg_interactive_budget/blob/main/fg_budget_classification_production.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Using DeBERTa, a Deep Learning model for detecting budget padding, in Nigeria's FG Budget Budget

In [1]:
# ============================================
# CELL 0a — IMPORTS
# ============================================

from sentence_transformers import CrossEncoder
import numpy as np
import pandas as pd

In [2]:
# ============================================
# CELL 0b — LOAD MODEL
# ============================================

model = CrossEncoder('abelakeni/pfmtools-deberta-v3-fg-budget-v3-classifier')
print("Model loaded successfully")



modules.json:   0%|          | 0.00/131 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/258 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/225 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/940 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.74G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/516 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Model loaded successfully


In [3]:
# ============================================
# CELL 1a — LOAD BUDGET FILE FOR ASSESSMENT
# ============================================
# Expected columns: code, project, mda_code, agency, amount


budget_df = pd.read_csv(
    'https://huggingface.co/datasets/abelakeni/fg-mda-objectives-2026-v1/resolve/main/correct_approved_2026_capital_budget.csv',
    # nrows=2000,
)

# Rename
# budget_df = budget_df.rename(columns={'mda_name_pdf': 'agency'})
# Drop rows with missing mandate or project text
budget_df = budget_df.dropna(subset=['ergp_line_item'])

print(f"Total projects loaded : {len(budget_df)}")
print(f"Agencies              : {budget_df['agency'].nunique()}")
print(f"Columns               : {list(budget_df.columns)}")

Total projects loaded : 36726
Agencies              : 852
Columns               : ['mda_code', 'agency', 'code', 'ergp_line_item', 'status', 'amount']


In [4]:
len(budget_df)

36726

In [5]:
# # # =====================================================
# # # ONE TIME ONLY - APPEND TO HUGGING FACE
# # # =====================================================

from huggingface_hub import HfApi

from google.colab import userdata
token = userdata.get('HF_TOKEN')

# # api = HfApi()
# # api.upload_file(
# #     path_or_fileobj='fg_agencies_objectives_2026_jan.csv',
# #     path_in_repo='fg_agencies_objectives_2026_jan.csv',
# #     repo_id='abelakeni/fg-mda-objectives-2026-v1',
# #     repo_type='dataset',
# #     token=token
# # )
# print("Uploaded!")


# # Budget data
# api = HfApi()
# api.upload_file(
#     path_or_fileobj='correct_approved_2026_capital_budget.csv',
#     path_in_repo='correct_approved_2026_capital_budget.csv',
#     repo_id='abelakeni/fg-mda-objectives-2026-v1',
#     repo_type='dataset',
#     token=token
# )
# print("Uploaded!")



In [7]:
# =====================================================
# CELL 1b — LOAD MDA OBJECTIVES MASTER FROM HUGGINGFACE
# =====================================================

# Master datasets has objectives of 876 federal MDAs only


import pandas as pd

mandate_df = pd.read_csv(
    'https://huggingface.co/datasets/abelakeni/fg-mda-objectives-2026-v1/resolve/main/fg_agencies_objectives_2026_jan.csv',
    # token='your-hf-token'  # only needed if repo is private
)




print(f"Mandate master loaded : {len(mandate_df)} agencies")
print(f"Columns               : {list(mandate_df.columns)}")
print("\n")
mandate_df.head(3)



Mandate master loaded : 876 agencies
Columns               : ['mda_code', 'mda_name', 'mandate_description']




,mda_code,mda_name,mandate_description
0,521025001,"41. COMMUNITY HEALTH TUTOR PROGRAMME, UCH",The Community Health Tutor Programme at the Un...
1,517015001,42. COMPUTER PROFESSIONALS (REGISTRATION COUNC...,The Computer Professionals (Registration Counc...
2,119009118,"43. CONSULATE GENERAL OF NIGERIA, FRANKFURT, G...",The Consulate General of the Federal Republic ...


In [11]:
# ============================================
# CELL 2 — COVERAGE DIAGNOSTICS
# ============================================

budget_codes  = set(budget_df['mda_code'].unique())
mandate_codes = set(mandate_df['mda_code'].unique())

# MDA codes in budget with NO mandate in master
unmatched_codes = budget_codes - mandate_codes

# MDA codes in master with NO projects in budget
unused_mandate_codes = mandate_codes - budget_codes

# Budget rows with no mandate coverage
unmatched_budget_df = budget_df[budget_df['mda_code'].isin(unmatched_codes)].copy()

# Mandate rows with no budget occurrence
unused_mandate_df = mandate_df[mandate_df['mda_code'].isin(unused_mandate_codes)].copy()

print("=" * 50)
print("COVERAGE DIAGNOSTICS")
print("=" * 50)
print(f"Unique MDA codes in budget              : {len(budget_codes)}")
print(f"Unique MDA codes in mandate master      : {len(mandate_codes)}")
print(f"MDA in Budget with no MDA mandate entry : {len(unmatched_codes)}")
print(f"Projects in Budget with no mandate      : {len(unmatched_budget_df)}")
print(f"Agencies in master, no budget projects  : {len(unused_mandate_codes)}")
print(f"Agencies in master, no budget projects  : {len(unused_mandate_df)}")

print("=" * 50)

print(f"\nBudget projects excluded from classification:")
print(unmatched_budget_df[['code', 'ergp_line_item', 'mda_code', 'agency']].head(10))

print(f"\nMandate agencies with no budget occurrence:")
print(unused_mandate_df[['mda_code', 'mda_name']].head(10))

COVERAGE DIAGNOSTICS
Unique MDA codes in budget              : 852
Unique MDA codes in mandate master      : 876
MDA in Budget with no MDA mandate entry : 15
Budget projects with no mandate         : 124
Agencies in master, no budget projects  : 39
Agencies in master, no budget projects  : 39

Budget projects excluded from classification:
              code                                     ergp_line_item  \
3591  ERGP20270035  PROVISION AND PROCUREMENT OF FARM IMPLEMENT,\n...   
3592  ERGP20270036  CONSTRUCTION OF SOLAR POWERED BOREHOLES ACROSS...   
3593  ERGP20270037  CONSTRUCTION OF BRIDGE AND EXPANSION OF GWAM M...   
3594  ERGP20270038  SURGICAL AND OPTHALMOLOGICAL OUTREACHES TO VUL...   
3595  ERGP20270039  CONSTRUCTION OF 2 AND 3 BEDROOMS HOUSING UNITS...   
3596  ERGP20270040  RENOVATION OF SOME SELECTED SCHOOLS IN\nAKWANG...   
3597  ERGP20270041  CONSTRUCTION OF BOREHOLES IN SOME SELECTED COM...   
3598  ERGP20270042  PROVISION OF HAND PUMP WATER BOREHOLES IN SEVE...   
35

In [12]:
unmatched_budget_df.tail()

,mda_code,agency,code,ergp_line_item,status,amount
36264,543001001,NATIONAL POPULATION COMMISSION,ERGP30111263,POPULATION ACTIVITY COORDINATION IN NIGERIA IN...,ONGOING,2.450000e+07
36265,543001001,NATIONAL POPULATION COMMISSION,ERGP30111271,SPECIALISED STUDIES IN DEMOGRAPHY AND OTHER PO...,ONGOING,3.500000e+07
36266,543001001,NATIONAL POPULATION COMMISSION,ERGP30111284,COMMEMORATION OF WORLD POPULATION DAY/ANNUAL P...,ONGOING,6.300000e+07
36267,543001001,NATIONAL POPULATION COMMISSION,ERGP30233764,PURCHASE OF OFFICIAL VEHICLES FOR HFCs,ONGOING,2.800000e+09
36268,543001001,NATIONAL POPULATION COMMISSION,ERGP7110825,"BUDGET PREPARATION AND IMPLEMENTATION, VERIFIC...",NaN,3.500000e+07


In [13]:
unused_mandate_df.head()

,mda_code,mda_name,mandate_description
10,515050001,21. BASIC HEALTH CARE PROVISION FUND,The Basic Health Care Provision Fund (BHCPF) w...
251,521028001,"242. FEDERAL STAFF CLINICS, ABUJA PHASE I","Federal Staff Clinics, Abuja Phase I, is a fed..."
252,521028002,"243. FEDERAL STAFF CLINICS, ABUJA PHASE II","Federal Staff Clinics, Abuja Phase II, is a fe..."
254,521028029,"245. FEDERAL STAFF DENTAL CLINICS, ABUJA","Federal Staff Dental Clinics, Abuja, is a spec..."
255,521028030,"246. FEDERAL STAFF DENTAL CLINICS, LAGOS","Federal Staff Dental Clinics, Lagos, is a spec..."


In [15]:
# ===================================================
# CELL 3 — BUILD CLEAN BUDGET + MAP MANDATE COLUMNS
# ===================================================

# Budget rows that have a mandate in master — ready for classification
clean_budget_df = budget_df[budget_df['mda_code'].isin(mandate_codes)].copy()

# Build lookup maps from mandate master
mandate_map  = mandate_df.set_index('mda_code')['mandate_description'].to_dict()
mda_name_map = mandate_df.set_index('mda_code')['mda_name'].to_dict()

# Map mandate and mda_name into clean budget
clean_budget_df['mda_name'] = clean_budget_df['mda_code'].map(mda_name_map)
clean_budget_df['mandate']  = clean_budget_df['mda_code'].map(mandate_map)

print(f"Total budget projects               : {len(budget_df)}")
print(f"Projects with mandate (clean)       : {len(clean_budget_df)}")
print(f"Projects excluded (no mandate)      : {len(unmatched_budget_df)}")
print(f"\nClean budget columns: {list(clean_budget_df.columns)}")

clean_budget_df[['mda_code', 'mda_name', 'ergp_line_item', 'mandate']].head(3)

Total budget projects               : 36726
Projects with mandate (clean)       : 36602
Projects excluded (no mandate)      : 124

Clean budget columns: ['mda_code', 'agency', 'code', 'ergp_line_item', 'status', 'amount', 'mda_name', 'mandate']


,mda_code,mda_name,ergp_line_item,mandate
0,111001001,844. STATE HOUSE HEADQUARTERS,PURCHASE OF SPORTING EQUIPMENT FOR STATE HOUSE...,State House Headquarters is presidential resid...
1,111001001,844. STATE HOUSE HEADQUARTERS,PROCUREMENT /MAINTENANCE OF EQUIPMENT FOR THE\...,State House Headquarters is presidential resid...
2,111001001,844. STATE HOUSE HEADQUARTERS,RENOVATION WORK ON 8 NO. BLOCKS OF 16 NO. 2 B/...,State House Headquarters is presidential resid...


In [17]:
# ============================================
# CELL 4 — PREPARE PAIRS
# ============================================

pairs = [
    (row['mandate'], row['ergp_line_item'])
    for idx, row in clean_budget_df.iterrows()
]

print(f"Total pairs prepared: {len(pairs)}")
print(f"\nSample pair:")
print(f"  Mandate : {pairs[3][0][:100]}...")
print(f"  Project : {pairs[3][1][:100]}...")

Total pairs prepared: 36602

Sample pair:
  Mandate : State House Headquarters is presidential residence and office complex in Abuja. Operates as seat of ...
  Project : CONSTRUCTION OF OFFICE COMPLEX FOR Sas and SSAs...


In [18]:
# ============================================
# CELL 5 — BATCH INFERENCE
# ============================================

batch_size = 32
all_predictions = []

for i in range(0, len(pairs), batch_size):
    batch = pairs[i:i + batch_size]
    preds = model.predict(batch, show_progress_bar=False)
    all_predictions.extend(preds)

    # Progress indicator
    if (i // batch_size) % 10 == 0:
        print(f"Processed {min(i + batch_size, len(pairs))}/{len(pairs)} projects")

all_predictions = np.array(all_predictions)
print(f"\nInference complete. Output shape: {all_predictions.shape}")

Processed 32/36602 projects
Processed 352/36602 projects
Processed 672/36602 projects
Processed 992/36602 projects
Processed 1312/36602 projects
Processed 1632/36602 projects
Processed 1952/36602 projects
Processed 2272/36602 projects
Processed 2592/36602 projects
Processed 2912/36602 projects
Processed 3232/36602 projects
Processed 3552/36602 projects
Processed 3872/36602 projects
Processed 4192/36602 projects
Processed 4512/36602 projects
Processed 4832/36602 projects
Processed 5152/36602 projects
Processed 5472/36602 projects
Processed 5792/36602 projects
Processed 6112/36602 projects
Processed 6432/36602 projects
Processed 6752/36602 projects
Processed 7072/36602 projects
Processed 7392/36602 projects
Processed 7712/36602 projects
Processed 8032/36602 projects
Processed 8352/36602 projects
Processed 8672/36602 projects
Processed 8992/36602 projects
Processed 9312/36602 projects
Processed 9632/36602 projects
Processed 9952/36602 projects
Processed 10272/36602 projects
Processed 1059

In [19]:
# ============================================
# CELL 6 — SOFTMAX + ASSIGN RESULTS
# ============================================

# Convert logits to probabilities
def softmax(x):
    exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))
    return exp_x / exp_x.sum(axis=1, keepdims=True)

probs = softmax(all_predictions)

# Add classification columns
clean_budget_df['predicted_class'] = np.argmax(all_predictions, axis=1)
clean_budget_df['prob_outside'] = probs[:, 0]   # class 0 = Outside Mandate
clean_budget_df['prob_within']  = probs[:, 1]   # class 1 = Within Mandate

# Human-readable label column
clean_budget_df['mandate_alignment'] = clean_budget_df['predicted_class'].map({
    0: 'Outside MDA Mandate',
    1: 'Within MDA Mandate'
})

# High confidence flag for priority review
clean_budget_df['flag_for_review'] = (
    (clean_budget_df['predicted_class'] == 0) &
    (clean_budget_df['prob_outside'] > 0.70)
)

print("Results assigned successfully")
print(clean_budget_df['mandate_alignment'].value_counts())
print(f"\nHigh-confidence violations flagged: {clean_budget_df['flag_for_review'].sum()}")

Results assigned successfully
mandate_alignment
Within MDA Mandate     18712
Outside MDA Mandate    17890
Name: count, dtype: int64

High-confidence violations flagged: 17861


In [20]:
# ============================================
# CELL 7 — REORDER COLUMNS + EXPORT
# ============================================

# Final column order matching your data structure
output_cols = [
    'code',
    'mda_code',
    'agency',
    'mda_name',
    'ergp_line_item',
    'status',
    'amount',
    'mandate_alignment',
    'predicted_class',
    'prob_outside',
    'prob_within',
    'flag_for_review'
]

# Only keep columns that exist (in case some are missing in your file)
output_cols = [c for c in output_cols if c in clean_budget_df.columns]

clean_budget_df_output = clean_budget_df[output_cols]

# Export
clean_budget_df_output.to_csv('correct_approved_capital_budget_2026_classified.csv', index=False)
print(f"Exported {len(clean_budget_df_output)} rows to correct_approved_capital_budget_2026_classified.csv")
print(f"\nColumn order in output:")
print(list(clean_budget_df_output.columns))

Exported 36602 rows to correct_approved_capital_budget_2026_classified.csv

Column order in output:
['code', 'mda_code', 'agency', 'mda_name', 'ergp_line_item', 'status', 'amount', 'mandate_alignment', 'predicted_class', 'prob_outside', 'prob_within', 'flag_for_review']


In [21]:
# ============================================
# CELL 8 — SUMMARY STATS
# ============================================

total = len(clean_budget_df_output)
outside = (clean_budget_df_output['predicted_class'] == 0).sum()
within  = (clean_budget_df_output['predicted_class'] == 1).sum()
flagged = clean_budget_df_output['flag_for_review'].sum()

print("=" * 50)
print("CLASSIFICATION SUMMARY")
print("=" * 50)
print(f"Total projects classified : {total}")
print(f"Within MDA Mandate        : {within}  ({within/total:.1%})")
print(f"Outside MDA Mandate       : {outside} ({outside/total:.1%})")
print(f"Flagged for review (>70%) : {flagged} ({flagged/total:.1%})")
print("=" * 50)

# Per-agency breakdown
agency_summary = clean_budget_df_output.groupby('agency').agg(
    total_projects=('predicted_class', 'count'),
    outside_mandate=('predicted_class', lambda x: (x == 0).sum()),
    flagged=('flag_for_review', 'sum')
).sort_values('outside_mandate', ascending=False)

agency_summary.reset_index(inplace=True)
agency_summary.to_csv('budget_2026_agency_summary.csv', index=False)

print("\nTop 10 agencies by outside-mandate projects:")
# agency_summary.head(10)

CLASSIFICATION SUMMARY
Total projects classified : 36602
Within MDA Mandate        : 18712  (51.1%)
Outside MDA Mandate       : 17890 (48.9%)
Flagged for review (>70%) : 17861 (48.8%)

Top 10 agencies by outside-mandate projects:


In [22]:
agency_summary.head(10)

,agency,total_projects,outside_mandate,flagged
0,FEDERAL CO-OPERATIVE COLLEGE- OJI RIVER,2791,2755,2755
1,NATIONAL PRODUCTIVITY CENTRE,1855,1815,1815
2,NIGERIAN BUILDING AND ROAD RESEARCH INSTITUTE ...,1382,1278,1277
3,"FEDERAL COLLEGE OF HORTICULTURE, DADIN-KOWA, G...",1296,1254,1253
4,FEDERAL CO-OPERATIVE COLLEGE- IBADAN,670,633,633
5,BORDER COMMUNITIES DEVELOPMENT AGENCY (BCDA) H...,723,491,480
6,NATIONAL CENTRE FOR AGRICULTURAL MECHANISATION...,462,415,415
7,AGRICULTURAL RESEARCH COUNCIL OF NIGERIA,324,299,299
8,FEDERAL COLLEGE OF AGRICULTURE - ISHIAGU,314,262,262
9,FEDERAL MINISTRY OF WORKS,1228,241,240
